# 01 Catalog Hierarchy

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Understand how Unity Catalog organizes data: <b>metastore → catalog → schema → objects</b> (tables, views, volumes, functions, models). Navigate it with SQL and Python, use three-level names, set your default catalog and schema, and query <code>information_schema</code> to discover what exists.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Every table you read or write on Databricks lives somewhere in this hierarchy, and every permission is granted on one of its levels. Governance is 15% of the Data Engineer Associate exam, and the hierarchy is the foundation for all of it: grants, lineage, managed storage and sharing.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 The Unity Catalog object model</b><br>
<code>`</code><br>Metastore  (one per cloud region per account; attached to workspaces)<br>└── Catalog            e.g. workspace, prod, dev, main          → top-level unit of isolation<br>    └── Schema         e.g. sales, bronze, governance             (also called a database)<br>        ├── Table      managed or external Delta (or other formats)<br>        ├── View       saved query; also materialized views<br>        ├── Volume     governed storage for files (CSV, JSON, images, ...)<br>        ├── Function   SQL or Python UDFs, row filters, column masks<br>        └── Model      registered ML models<br><code>`</code>
<ul><li>Objects are addressed by a <b>three-level name</b>: <code>catalog.schema.object</code>, for example <code>workspace.governance.customers</code></li><li>Outside the metastore hierarchy, the metastore also holds <b>storage credentials</b>, <b>external locations</b>, <b>connections</b> and <b>shares</b> (Delta Sharing)</li><li><code>hive_metastore</code> is the legacy per-workspace metastore, shown as a catalog for compatibility. New work goes into Unity Catalog catalogs</li><li>In Free Edition, your workspace comes with a catalog called **<code>workspace</code>** with a <code>default</code> schema</li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A retail company uses one metastore for its EU region with catalogs per environment and domain: <code>dev_sales</code>, <code>prod_sales</code>, <code>prod_finance</code>. Inside each, schemas follow the medallion layers (<code>bronze</code>, <code>silver</code>, <code>gold</code>). Analysts get <code>USE CATALOG</code> on <code>prod_sales</code> and <code>SELECT</code> on <code>gold</code> only. Engineers in dev can create tables in <code>dev_sales</code>. The same code runs in both environments by changing one parameter: the catalog name.
</div>

## Syntax

```sql
SHOW CATALOGS;                                 -- catalogs you can see
SHOW SCHEMAS IN workspace;                     -- schemas in a catalog
SHOW TABLES IN workspace.default;              -- tables and views in a schema
USE CATALOG workspace; USE SCHEMA governance;  -- set defaults for unqualified names
SELECT current_catalog(), current_schema();    -- where am I?
SELECT * FROM workspace.governance.customers;  -- three-level name (works from anywhere)
```

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates the schema used by this folder, <code>governance</code>, and a small <code>customers</code> table in it.<br><br>
<b>Why it matters</b><br>In Unity Catalog, <code>CREATE SCHEMA</code> creates a container inside a catalog. You need the <code>CREATE SCHEMA</code> privilege on the catalog (you have it on <code>workspace</code> in Free Edition).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
No output. The schema and table exist.
</div>

In [0]:
CATALOG = "workspace"
DB = f"{CATALOG}.governance"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB} COMMENT 'Lessons of 06_unity_catalog'")

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {DB}.customers (customer_id BIGINT, name STRING, email STRING, country STRING)
""")
spark.sql(f"""
    INSERT INTO {DB}.customers VALUES
      (1, 'Asha',  'asha@example.com',  'IN'),
      (2, 'Ben',   'ben@example.com',   'UK'),
      (3, 'Chen',  'chen@example.com',  'AE'),
      (4, 'Divya', 'divya@example.com', 'IN')
""")

## 1. Where am I?

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows the current catalog and schema, and the catalogs you can see.<br><br>
<b>Why it matters</b><br>Unqualified names (<code>SELECT * FROM customers</code>) resolve against the <b>current</b> catalog and schema. Knowing them avoids writing to the wrong place.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The current catalog (<code>workspace</code> on Databricks) and schema (<code>default</code>), then a list of catalogs. On Free Edition you'll typically see <code>workspace</code>, <code>system</code> and <code>samples</code>.
</div>

In [0]:
spark.sql("SELECT current_catalog() AS catalog, current_schema() AS schema").show()
spark.sql("SHOW CATALOGS").show()

## 2. Walk down the hierarchy

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists the schemas in your catalog, then the tables in the <code>governance</code> schema, with SQL and with the Python catalog API.<br><br>
<b>Why it matters</b><br><code>SHOW</code> commands are quick for exploring. <code>spark.catalog</code> is handy in Python code, for example to check whether a table exists before creating it.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The schemas include <code>default</code> and <code>governance</code>. The <code>governance</code> schema contains <code>customers</code>, and <code>tableExists</code> returns <code>True</code>.
</div>

In [0]:
spark.sql(f"SHOW SCHEMAS IN {CATALOG}").show()
spark.sql(f"SHOW TABLES IN {DB}").show()
print("customers exists:", spark.catalog.tableExists(f"{DB}.customers"))

## 3. Three-level names and defaults

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads the same table with a full three-level name, then sets the default catalog and schema and reads it with just the table name.<br><br>
<b>Why it matters</b><br>Production code should use <b>full names</b>, or a catalog parameter, so that it doesn't depend on whatever the current schema happens to be. <code>USE</code> is convenient interactively.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Both queries return 4 rows.
</div>

In [0]:
print("three-level name:", spark.table(f"{DB}.customers").count())

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("USE SCHEMA governance")

In [0]:
print("after USE, unqualified name:", spark.table("customers").count(), "| current schema:", spark.catalog.currentDatabase())

## 4. Parameterize the catalog for environments

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes a function that builds table names from an environment, so the same code targets <code>dev</code> or <code>prod</code> catalogs.<br><br>
<b>Why it matters</b><br>Hardcoding <code>prod.sales.orders</code> in notebooks makes testing in dev dangerous. A catalog parameter (a widget, a job parameter or a bundle variable) is the standard pattern.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Different full names for <code>dev</code> and <code>prod</code>, and the <code>workspace</code> catalog used for this course.
</div>

In [0]:
def table_name(env, schema, table):
    catalog = {"dev": "dev_sales", "prod": "prod_sales", "course": CATALOG}[env]
    return f"{catalog}.{schema}.{table}"

for env in ["dev", "prod", "course"]:
    print(f"{env:<7} -> {table_name(env, 'governance', 'customers')}")
print("course table rows:", spark.table(table_name("course", "governance", "customers")).count())

## 5. Discover objects with `information_schema`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Queries the catalog's <code>information_schema</code> for the tables and columns in the <code>governance</code> schema.<br><br>
<b>Why it matters</b><br>Every Unity Catalog catalog has an <code>information_schema</code> with views over its metadata (tables, columns, views, volumes, privileges, tags). It's how you answer questions like "which tables have a column called <code>email</code>?" with SQL.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Works in Free Edition. <code>system.information_schema</code> covers all catalogs in the metastore.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The <code>customers</code> table with its type, and its four columns with data types.
</div>

In [0]:
display(spark.sql(f"""
    SELECT table_name, table_type, created_by, created
    FROM {CATALOG}.information_schema.tables
    WHERE table_schema = 'governance'
"""))
display(spark.sql(f"""
    SELECT table_name, column_name, data_type, ordinal_position
    FROM {CATALOG}.information_schema.columns
    WHERE table_schema = 'governance' AND table_name = 'customers'
    ORDER BY ordinal_position
"""))

## 6. Find columns by name across schemas

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Searches <code>information_schema.columns</code> for every column whose name looks like personal data.<br><br>
<b>Why it matters</b><br>Governance work often starts with an inventory: where is PII stored? This query is the first step towards tagging and masking those columns (lessons 05 and 07).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The <code>email</code> and <code>name</code> columns of <code>customers</code> (and any similar columns in your other schemas).
</div>

In [0]:
display(spark.sql(f"""
    SELECT table_schema, table_name, column_name
    FROM {CATALOG}.information_schema.columns
    WHERE lower(column_name) RLIKE 'email|phone|name|address|ssn|dob'
      AND table_schema <> 'information_schema'
    ORDER BY table_schema, table_name
"""))

## Under the hood

```
query "SELECT * FROM customers"
  └─ name resolution: current_catalog() . current_schema() . customers  → workspace.governance.customers
       └─ Unity Catalog checks privileges: USE CATALOG workspace → USE SCHEMA governance → SELECT on table
            └─ returns the table's storage location + short-lived credentials to the compute
                 └─ compute reads the Delta files
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Metadata</span> `SHOW`, `DESCRIBE` and `information_schema` queries read the metastore, not the data files, so they're fast.

In [0]:
spark.sql(f"DESCRIBE TABLE EXTENDED {DB}.customers").filter("col_name IN ('Catalog', 'Database', 'Table', 'Type', 'Provider')").show(truncate=False)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>TABLE_OR_VIEW_NOT_FOUND</code> for a table that exists</b><br>
The current catalog or schema is different from what you assumed. Use the full three-level name, or check <code>current_catalog()</code> and <code>current_schema()</code>.<br><br>
**⛔ Problem: <code>PERMISSION_DENIED</code> on a table you can see in the list**<br>You need <code>USE CATALOG</code> and <code>USE SCHEMA</code> on the parents <b>and</b> <code>SELECT</code> on the table. See lesson 04.<br><br>
**⛔ Problem: tables created in <code>hive_metastore</code> by old code**<br>Legacy code or a <code>spark_catalog</code> default. New tables belong in Unity Catalog catalogs: use full names.<br><br>
**⛔ Problem: hardcoded <code>prod</code> catalog in notebooks**<br>Pass the catalog as a parameter so code can run in dev and prod.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. Describe the Unity Catalog object hierarchy.</b><br>
A metastore is the top-level container, usually one per region, attached to workspaces. It contains catalogs, catalogs contain schemas, and schemas contain tables, views, volumes, functions and models. Objects are referenced with three-level names: <code>catalog.schema.object</code>. The metastore also holds storage credentials, external locations, connections and shares.<br><br>

<b>🎤 2. What is the difference between a catalog and a schema?</b><br>
A catalog is the top-level grouping, typically used to separate environments or business units, and is the main boundary for isolation and access. A schema (database) groups related tables, views, volumes and functions inside a catalog, for example a medallion layer or a domain.<br><br>

<b>🎤 3. What is <code>hive_metastore</code> in a Unity Catalog workspace?</b><br>
The workspace's legacy Hive metastore, exposed as a catalog for backward compatibility. It doesn't get Unity Catalog governance features like centralized grants and lineage, so new data should go into Unity Catalog catalogs.<br><br>

<b>🎤 4. How do you find all tables containing a column named <code>email</code>?</b><br>
Query <code>information_schema.columns</code> in the catalog (or <code>system.information_schema.columns</code> for all catalogs) filtered on <code>column_name</code>.<br><br>

<b>🎤 5. Why use three-level names in production code?</b><br>
They make the target explicit and independent of the session's current catalog and schema, which avoids reading from or writing to the wrong environment. The catalog part is usually a parameter.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. What is the correct order of the Unity Catalog object hierarchy, from top to bottom?</b><br>
A. Catalog → Metastore → Schema → Table<br>
B. Metastore → Catalog → Schema → Table<br>
C. Metastore → Schema → Catalog → Table<br>
D. Workspace → Table → Schema → Catalog<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. A query <code>SELECT * FROM orders</code> fails with "table not found", but <code>prod.sales.orders</code> exists. What is the most likely cause?</b><br>
A. The table is not a Delta table<br>
B. The session's current catalog or schema is not <code>prod.sales</code><br>
C. Unity Catalog doesn't support unqualified names<br>
D. The table has no comment<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q3. Which object is NOT stored inside a schema in Unity Catalog?</b><br>
A. Volume<br>
B. Function<br>
C. External location<br>
D. View<br>
<details><summary><b>Show answer</b></summary><b>C.</b> External locations (and storage credentials) are metastore-level objects.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create a schema <code>governance_practice</code> in your catalog, with a comment</li><li>Create a table <code>products (product_id BIGINT, name STRING, price DOUBLE)</code> in it using a three-level name, and insert 3 rows</li><li>With <code>information_schema</code>, list all tables in <code>governance</code> and <code>governance_practice</code> with their column counts</li><li>Drop the practice schema with <code>CASCADE</code></li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Schema with a comment
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.governance_practice COMMENT 'Practice schema'")

# 2. Table with a three-level name
spark.sql(f"CREATE OR REPLACE TABLE {CATALOG}.governance_practice.products (product_id BIGINT, name STRING, price DOUBLE)")
spark.sql(f"INSERT INTO {CATALOG}.governance_practice.products VALUES (1, 'Pen', 1.5), (2, 'Book', 12.0), (3, 'Bag', 30.0)")
print("products:", spark.table(f"{CATALOG}.governance_practice.products").count())

In [0]:
# 3. Column counts per table (Databricks: information_schema)
display(spark.sql(f"""
    SELECT table_schema, table_name, COUNT(*) AS columns
    FROM {CATALOG}.information_schema.columns
    WHERE table_schema IN ('governance', 'governance_practice')
    GROUP BY table_schema, table_name ORDER BY 1, 2
"""))

In [0]:
# 4. Drop the schema and everything in it
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.governance_practice CASCADE")
print("practice schema dropped")

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Metastore → catalog → schema → table / view / volume / function / model</li><li>Three-level names: <code>catalog.schema.object</code>. Use them (with a catalog parameter) in production code</li><li><code>USE CATALOG</code> / <code>USE SCHEMA</code> set defaults for unqualified names</li><li><code>information_schema</code> (per catalog) and <code>system.information_schema</code> (all catalogs) describe everything with SQL</li><li>Storage credentials, external locations, connections and shares are metastore-level objects</li><li><code>hive_metastore</code> is legacy. Build new things in Unity Catalog</li></ul>
</div>

| Task | Command |
|---|---|
| Where am I? | `SELECT current_catalog(), current_schema()` |
| List | `SHOW CATALOGS`, `SHOW SCHEMAS IN c`, `SHOW TABLES IN c.s` |
| Set defaults | `USE CATALOG c`, `USE SCHEMA s` |
| Exists? | `spark.catalog.tableExists("c.s.t")` |
| Search metadata | `c.information_schema.tables / columns` |
| Create / drop schema | `CREATE SCHEMA c.s`, `DROP SCHEMA c.s CASCADE` |

## Git commit

```
git add 06_unity_catalog/01_catalog_hierarchy.ipynb
git commit -m "add 06_01 catalog hierarchy notebook"
```